# FLX-Nano — Surgical Transplant & Retrain

Load the Phase 5 model, transplant specialist donor model layers into dimension-agnostic cortices,
fine-tune the interfaces, then retrain Phase 2 (thermal) and Phase 5 (hypothesis) to fix validation failures.

## Donor Models

| Cortex | Donor | hidden_size | Layers |
|---|---|---|---|
| language | SmolLM2-1.7B | 2048 | 24 |
| math | Qwen2.5-Math-1.5B | 1536 | 28 |
| code | DeepSeek-Coder-1.3B | 2048 | 24 |
| science | Galactica-1.3B | 2048 | 24 |
| reasoning | Phi-2 | 2560 | 32 |

**Runtime → Change runtime type → GPU (A100 preferred)**

In [ ]:
# Cell 1: Setup — clone, install, mount Drive
!git clone https://github.com/Unseengap/flux-model.git
%cd flux-model
!pip install -e '.[dev]' -q
!pip install datasets transformers accelerate -q

In [ ]:
# Cell 2: Mount Drive + GPU check
from google.colab import drive
drive.mount('/content/drive')

FLX_HUB = '/content/drive/MyDrive/flx_state'

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Cell 3: Load the completed Phase 5 model
from flx.serialization import save_flx, load_flx
from flx.model import FLXNano
from flx.hypothesis import HypothesisHead
from flx.memory import EpisodicBuffer, EpisodicCompressor

old_model, episodic_buffer, activation_history = load_flx(
    f'{FLX_HUB}/nano_phase5.flx', device=DEVICE
)
old_model.eval()

# Re-attach hypothesis head from Phase 5 checkpoint (not serialized yet)
import os
hyp_ckpt = f'{FLX_HUB}/checkpoints/phase5_epoch4.pt'
if os.path.exists(hyp_ckpt):
    hypothesis_head = HypothesisHead(d_model=512, hypothesis_dim=512).to(DEVICE)
    ckpt = torch.load(hyp_ckpt, map_location=DEVICE, weights_only=True)
    state = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    hypothesis_head.load_state_dict(state)
    old_model.attach_hypothesis_head(hypothesis_head)
    print('\u2713 HypothesisHead loaded from Phase 5 checkpoint')

counts = old_model.count_parameters()
print(f'\nOld model loaded — {sum(counts.values()):,} total parameters')
for k, v in counts.items():
    print(f'  {k}: {v:,}')

In [ ]:
# Cell 4: Define donor model registry
from copy import deepcopy
from transformers import AutoModelForCausalLM, AutoConfig
import torch.nn as nn

DONORS = {
    'language': {
        'hf_id': 'HuggingFaceTB/SmolLM2-1.7B',
        'hidden_size': 2048,
        'num_layers': 24,
        'arch': 'llama',
        'layer_picks': {'intermediate': (4, 5), 'expert': (10, 11), 'frontier': (16, 17)},
    },
    'math': {
        'hf_id': 'Qwen/Qwen2.5-Math-1.5B',
        'hidden_size': 1536,
        'num_layers': 28,
        'arch': 'qwen2',
        'layer_picks': {'intermediate': (5, 6), 'expert': (12, 13), 'frontier': (19, 20)},
    },
    'code': {
        'hf_id': 'deepseek-ai/deepseek-coder-1.3b-base',
        'hidden_size': 2048,
        'num_layers': 24,
        'arch': 'llama',
        'layer_picks': {'intermediate': (4, 5), 'expert': (10, 11), 'frontier': (16, 17)},
    },
    'science': {
        'hf_id': 'facebook/galactica-1.3b',
        'hidden_size': 2048,
        'num_layers': 24,
        'arch': 'opt',
        'layer_picks': {'intermediate': (4, 5), 'expert': (10, 11), 'frontier': (16, 17)},
    },
    'reasoning': {
        'hf_id': 'microsoft/phi-2',
        'hidden_size': 2560,
        'num_layers': 32,
        'arch': 'phi',
        'layer_picks': {'intermediate': (6, 7), 'expert': (14, 15), 'frontier': (22, 23)},
    },
}

print('Donor registry:')
for name, cfg in DONORS.items():
    print(f'  {name:>12}: {cfg["hf_id"]} (d={cfg["hidden_size"]}, L={cfg["num_layers"]}, arch={cfg["arch"]})')

In [ ]:
# Cell 5: Define DonorStratumLayers — wraps HuggingFace decoder layers
# as a drop-in replacement for nn.TransformerEncoder

class DonorStratumLayers(nn.Module):
    """Wraps a list of HuggingFace decoder layers to drop in for nn.TransformerEncoder.

    Handles Llama, Qwen2, OPT, and Phi architectures.
    """

    def __init__(self, layers: nn.ModuleList, arch: str, hidden_size: int):
        super().__init__()
        self.donor_layers = layers
        self.arch = arch
        self.hidden_size = hidden_size

    @property
    def num_layers(self):
        return len(self.donor_layers)

    def forward(self, src, mask=None, src_key_padding_mask=None):
        """Forward through donor layers.

        Args:
            src: [batch, seq, hidden_size]
            mask: Ignored (donor layers handle causal masking internally).

        Returns:
            [batch, seq, hidden_size]
        """
        hidden = src
        batch, seq_len = hidden.shape[0], hidden.shape[1]

        if self.arch in ('llama', 'qwen2'):
            # Llama/Qwen2 decoder layers: (hidden_states, attention_mask, position_ids)
            position_ids = torch.arange(seq_len, device=hidden.device).unsqueeze(0).expand(batch, -1)
            # Causal mask: HF layers handle this via causal_mask internally
            for layer in self.donor_layers:
                out = layer(hidden, position_ids=position_ids)
                hidden = out[0] if isinstance(out, tuple) else out

        elif self.arch == 'opt':
            # OPT decoder layers: (hidden_states, attention_mask)
            # OPT uses a 2D attention mask of shape [batch, seq]
            attention_mask = torch.ones(batch, seq_len, device=hidden.device)
            for layer in self.donor_layers:
                out = layer(hidden, attention_mask=attention_mask)
                hidden = out[0] if isinstance(out, tuple) else out

        elif self.arch == 'phi':
            # Phi decoder layers: (hidden_states, attention_mask, position_ids)
            position_ids = torch.arange(seq_len, device=hidden.device).unsqueeze(0).expand(batch, -1)
            for layer in self.donor_layers:
                out = layer(hidden, position_ids=position_ids)
                hidden = out[0] if isinstance(out, tuple) else out

        else:
            raise RuntimeError(f'Unknown donor architecture: {self.arch}')

        return hidden


def extract_donor_layers(model, arch, layer_indices):
    """Extract specific decoder layers from a HuggingFace model.

    Args:
        model: HuggingFace causal LM.
        arch: Architecture type ('llama', 'qwen2', 'opt', 'phi').
        layer_indices: Tuple of (start, end) layer indices (inclusive).

    Returns:
        nn.ModuleList of deepcopied decoder layers.
    """
    start, end = layer_indices
    if arch in ('llama', 'qwen2'):
        all_layers = model.model.layers
    elif arch == 'opt':
        all_layers = model.model.decoder.layers
    elif arch == 'phi':
        all_layers = model.model.layers
    else:
        raise RuntimeError(f'Unknown architecture: {arch}')

    selected = nn.ModuleList([deepcopy(all_layers[i]) for i in range(start, end + 1)])
    return selected


print('DonorStratumLayers and extract_donor_layers defined')

In [ ]:
# Cell 6: Build new model with per-cortex dimensions
from flx.model import FLXNano, DomainCortex
from flx.router import ThalamicRouter
from flx.thermal import ThermalEstimator
from flx.bridges import build_bridges
from flx.memory import MemoryController
from flx.meta_gen import MetaDeltaGenerator

cortex_names = old_model.cortex_names
cortex_dims = {
    'language': 2048,
    'math': 1536,
    'code': 2048,
    'science': 2048,
    'reasoning': 2560,
}

# Read config from old model's manifest
d_model = old_model.d_model
new_model = FLXNano(
    vocab_size=old_model.shared_trunk.token_embedding.num_embeddings,
    d_model=d_model,
    nhead=old_model.shared_trunk.trunk_layers.layers[0].self_attn.num_heads,
    trunk_layers=old_model.shared_trunk.trunk_layers.num_layers,
    layers_per_stratum=2,  # placeholder — will be replaced by donor layers
    cortex_names=cortex_names,
    cortex_dims=cortex_dims,
    delta_rank=old_model.delta_rank,
    delta_capacity=8,
    max_seq_len=old_model.shared_trunk.position_embedding.num_embeddings,
    dim_feedforward=old_model.shared_trunk.trunk_layers.layers[0].linear1.out_features,
).to(DEVICE)

# Copy non-cortex weights from old model
new_model.shared_trunk.load_state_dict(old_model.shared_trunk.state_dict())
new_model.cortex_merger.load_state_dict(old_model.cortex_merger.state_dict())
new_model.decoder.load_state_dict(old_model.decoder.state_dict())

# Copy router
if old_model.thalamic_router is not None:
    router = ThalamicRouter(d_model=d_model, cortex_names=cortex_names).to(DEVICE)
    router.load_state_dict(old_model.thalamic_router.state_dict())
    new_model.attach_router(router)

# Copy bridges
if old_model.bridges is not None:
    bridges = build_bridges(cortex_names, d_model=d_model).to(DEVICE)
    bridges.load_state_dict(old_model.bridges.state_dict())
    new_model.attach_bridges(bridges)

# Copy memory controller
if old_model.memory_controller is not None:
    mc = MemoryController(d_model=d_model, episode_dim=256).to(DEVICE)
    mc.load_state_dict(old_model.memory_controller.state_dict())
    new_model.attach_memory(mc)

# Copy meta generator
if old_model.meta_generator is not None:
    mg = MetaDeltaGenerator(d_model=d_model, delta_rank=old_model.delta_rank,
                            num_cortices=len(cortex_names)).to(DEVICE)
    mg.load_state_dict(old_model.meta_generator.state_dict())
    new_model.attach_meta_generator(mg)

# Verify cortex dims
for name, cortex in new_model.cortices.items():
    print(f'  {name}: internal_dim={cortex.internal_dim}, '
          f'has_adapters={not isinstance(cortex.proj_in, nn.Identity)}')

print(f'\nNew model structure ready — cortex strata still have placeholder weights')

# Free old model
del old_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Cell 7: Transplant — load each donor, extract layers, replace strata
import gc

for cortex_name, donor_cfg in DONORS.items():
    print(f'\n=== Transplanting {cortex_name} from {donor_cfg["hf_id"]} ===')

    # Load donor in fp16 to save memory
    donor = AutoModelForCausalLM.from_pretrained(
        donor_cfg['hf_id'],
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    donor.eval()
    print(f'  Loaded {donor_cfg["hf_id"]} ({sum(p.numel() for p in donor.parameters())/1e6:.0f}M params)')

    cortex = new_model.cortices[cortex_name]
    arch = donor_cfg['arch']
    hidden_size = donor_cfg['hidden_size']

    for stratum_name, layer_range in donor_cfg['layer_picks'].items():
        stratum = cortex.strata[stratum_name]

        # Extract and wrap donor layers
        extracted = extract_donor_layers(donor, arch, layer_range)
        wrapper = DonorStratumLayers(extracted, arch, hidden_size)

        # Replace the stratum's transformer layers
        stratum.layers = wrapper.to(DEVICE).to(torch.float32)

        # Clear incompatible delta stack
        from flx.delta import DeltaStack
        stratum.delta_stack = DeltaStack(capacity=8)

        print(f'  {stratum_name}: layers {layer_range[0]}-{layer_range[1]} '
              f'({wrapper.num_layers} layers, {sum(p.numel() for p in wrapper.parameters())/1e6:.1f}M params)')

    # Delete donor to free GPU memory
    del donor
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Summary
counts = new_model.count_parameters()
print(f'\nTransplanted model — {sum(counts.values()):,} total parameters')
for k, v in counts.items():
    print(f'  {k}: {v:,}')

In [ ]:
# Cell 8: Phase 0.5a — Adapter + Interface Fine-Tune
# Freeze all donor cortex layers. Train only adapters, router, merger, decoder.
import numpy as np
import torch.nn.functional as F
from collections import defaultdict
from transformers import AutoTokenizer
from datasets import load_dataset
import itertools, time

# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-hf')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
SEQ_LEN = 512

def tokenize_text(text, max_len=SEQ_LEN):
    ids = tokenizer(text, truncation=True, max_length=max_len + 1,
                    return_tensors='pt')['input_ids'][0]
    if len(ids) < 2:
        return None, None
    input_ids = ids[:-1]
    targets = ids[1:]
    pad_len = max_len - len(input_ids)
    if pad_len > 0:
        input_ids = F.pad(input_ids, (0, pad_len), value=tokenizer.pad_token_id)
        targets = F.pad(targets, (0, pad_len), value=-100)
    return input_ids.unsqueeze(0).to(DEVICE), targets.unsqueeze(0).to(DEVICE)

# --- Load domain data (50 samples per domain) ---
N_PER_DOMAIN = 50
domain_texts = {}

print('Loading domain data...')
ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
domain_texts['language'] = [x['text'][:500] for x in itertools.islice(ds, N_PER_DOMAIN)]
print(f'  language: {len(domain_texts["language"])}')

ds = load_dataset('openai/gsm8k', 'main', split='test')
domain_texts['math'] = [f"Problem: {x['question']}\nSolution: {x['answer']}" for x in ds][:N_PER_DOMAIN]
print(f'  math: {len(domain_texts["math"])}')

ds = load_dataset('code_search_net', 'python', split='test', streaming=True)
domain_texts['code'] = [x['whole_func_string'][:500] for x in itertools.islice(ds, N_PER_DOMAIN)]
print(f'  code: {len(domain_texts["code"])}')

ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test')
domain_texts['science'] = [f"Question: {x['question']}\nChoices: {', '.join(x['choices']['text'])}" for x in ds][:N_PER_DOMAIN]
print(f'  science: {len(domain_texts["science"])}')

ds = load_dataset('cais/mmlu', 'all', split='test', streaming=True)
domain_texts['reasoning'] = [f"Question: {x['question']}\nChoices: {', '.join(x['choices'])}" for x in itertools.islice(ds, N_PER_DOMAIN)]
print(f'  reasoning: {len(domain_texts["reasoning"])}')

# Build training set
all_samples = []
for domain, texts in domain_texts.items():
    for text in texts:
        inp, tgt = tokenize_text(text)
        if inp is not None:
            all_samples.append((inp, tgt))
print(f'\nTotal training samples: {len(all_samples)}')

# --- Phase 0.5a: Freeze donor layers, train adapters + interface ---
new_model.train()

# Freeze EVERYTHING first
for p in new_model.parameters():
    p.requires_grad = False

# Unfreeze: adapters, router, merger, decoder, confidence, difficulty_gate
for cortex in new_model.cortices.values():
    if hasattr(cortex, 'proj_in') and not isinstance(cortex.proj_in, nn.Identity):
        for p in cortex.proj_in.parameters():
            p.requires_grad = True
        for p in cortex.proj_out.parameters():
            p.requires_grad = True
    for p in cortex.difficulty_gate.parameters():
        p.requires_grad = True
    for stratum in cortex.strata.values():
        stratum.confidence.requires_grad = True

if new_model.thalamic_router is not None:
    for p in new_model.thalamic_router.parameters():
        p.requires_grad = True
for p in new_model.cortex_merger.parameters():
    p.requires_grad = True
for p in new_model.decoder.parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in new_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in new_model.parameters())
print(f'Phase 0.5a: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)')

optimizer = torch.optim.AdamW(
    [p for p in new_model.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01
)

# Training loop
NUM_STEPS = 1500
GRAD_ACCUM = 4
rng = np.random.default_rng(42)
losses_05a = []

print(f'\nPhase 0.5a: {NUM_STEPS} steps (adapters + interface frozen donor layers)')
for step in range(NUM_STEPS):
    idx = rng.integers(0, len(all_samples))
    inp, tgt = all_samples[idx]

    logits = new_model(inp)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt.view(-1), ignore_index=-100)
    (loss / GRAD_ACCUM).backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(
            [p for p in new_model.parameters() if p.requires_grad], 1.0
        )
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    losses_05a.append(loss.item())
    if step % 200 == 0:
        avg = np.mean(losses_05a[-200:]) if len(losses_05a) >= 200 else np.mean(losses_05a)
        print(f'  step {step:4d}/{NUM_STEPS} | loss={avg:.4f}')

print(f'Phase 0.5a done. Final avg loss: {np.mean(losses_05a[-200:]):.4f}')

In [ ]:
# Cell 9: Phase 0.5b — Full Fine-Tune (low LR for donor layers)
# Unfreeze everything. Donor layers get low LR, adapters/interface get higher LR.

# Build two param groups
donor_params = []
interface_params = []
for name, cortex in new_model.cortices.items():
    for sname, stratum in cortex.strata.items():
        for p in stratum.layers.parameters():
            p.requires_grad = True
            donor_params.append(p)

for p in new_model.parameters():
    if p.requires_grad and not any(p is dp for dp in donor_params):
        interface_params.append(p)

optimizer_b = torch.optim.AdamW([
    {'params': donor_params, 'lr': 1e-5},
    {'params': interface_params, 'lr': 3e-5},
], weight_decay=0.01)

donor_trainable = sum(p.numel() for p in donor_params)
interface_trainable = sum(p.numel() for p in interface_params)
print(f'Phase 0.5b: donor={donor_trainable:,} (lr=1e-5) + interface={interface_trainable:,} (lr=3e-5)')

NUM_STEPS_B = 2500
losses_05b = []

print(f'\nPhase 0.5b: {NUM_STEPS_B} steps (full fine-tune, low LR for donors)')
for step in range(NUM_STEPS_B):
    idx = rng.integers(0, len(all_samples))
    inp, tgt = all_samples[idx]

    logits = new_model(inp)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt.view(-1), ignore_index=-100)
    (loss / GRAD_ACCUM).backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(
            donor_params + interface_params, 1.0
        )
        optimizer_b.step()
        optimizer_b.zero_grad(set_to_none=True)

    losses_05b.append(loss.item())
    if step % 300 == 0:
        avg = np.mean(losses_05b[-300:]) if len(losses_05b) >= 300 else np.mean(losses_05b)
        print(f'  step {step:4d}/{NUM_STEPS_B} | loss={avg:.4f}')

print(f'Phase 0.5b done. Final avg loss: {np.mean(losses_05b[-300:]):.4f}')

In [ ]:
# Cell 10: Quick validation — verify non-degenerate output
new_model.eval()

test_prompts = [
    'Once upon a time in a distant kingdom, there lived a',
    'Prove that the sum of two even numbers is always even.',
    'def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n',
    'The process of photosynthesis converts carbon dioxide and water into',
    'If all cats are animals and all animals need water, then all cats',
]

print('=== Quick Validation: Greedy Decode ===')
with torch.no_grad():
    for prompt in test_prompts:
        inp, _ = tokenize_text(prompt)
        if inp is None:
            continue

        # Get tau and routing
        trunk_out = new_model.shared_trunk(inp)
        tau = new_model.thermal_estimator(trunk_out).mean().item() if new_model.thermal_estimator else 0.5
        scores = new_model.thalamic_router(trunk_out) if new_model.thalamic_router else {}
        top_cortex = max(scores, key=lambda k: scores[k].mean().item()) if scores else 'none'

        # Greedy decode 50 tokens
        generated = inp[0].tolist()
        gen_input = inp.clone()
        for _ in range(50):
            out = new_model(gen_input)
            next_token = out[0, -1].argmax().item()
            generated.append(next_token)
            gen_input = torch.tensor([generated[-SEQ_LEN:]], device=DEVICE)
        decoded = tokenizer.decode(generated, skip_special_tokens=True)

        # Check for degenerate repetition
        tokens_tail = generated[-30:]
        trigrams = [tuple(tokens_tail[j:j+3]) for j in range(len(tokens_tail)-2)]
        unique_ratio = len(set(trigrams)) / max(len(trigrams), 1)
        status = '\u2713' if unique_ratio >= 0.3 else '\u2717 DEGENERATE'

        continuation = decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded
        print(f'\n  {status} [\u03c4={tau:.3f}, top={top_cortex}]')
        print(f'  PROMPT: {prompt[:60]}...')
        print(f'  MODEL:  {continuation[:120]}')

---
## Phase 2 Retrain — Fix Thermal Inversion

Reinitialize ThermalEstimator from scratch. Now that cortices produce meaningful loss differences on easy vs hard inputs, the bidirectional τ target has real gradient signal.

In [ ]:
# Cell 11: Phase 2 Retrain — Fix Thermal Efficiency
from flx.thermal import ThermalEstimator
from flx.training.phase2_thermal import phase2_training_step

# Reinitialize thermal estimator from scratch
fresh_thermal = ThermalEstimator(d_model=d_model).to(DEVICE)
new_model.attach_thermal(fresh_thermal)
new_model.train()

# Freeze shared trunk and cortex stratum layers (only train thermal + bridges + gates)
for p in new_model.shared_trunk.parameters():
    p.requires_grad = False
for cortex in new_model.cortices.values():
    for stratum in cortex.strata.values():
        for p in stratum.layers.parameters():
            p.requires_grad = False
    # Keep adapters, confidence, difficulty_gate trainable
    if hasattr(cortex, 'proj_in') and not isinstance(cortex.proj_in, nn.Identity):
        for p in cortex.proj_in.parameters():
            p.requires_grad = True
        for p in cortex.proj_out.parameters():
            p.requires_grad = True
    cortex.difficulty_gate.requires_grad_(True)
    for stratum in cortex.strata.values():
        stratum.confidence.requires_grad = True

# Thermal + bridges always trainable
for p in new_model.thermal_estimator.parameters():
    p.requires_grad = True
if new_model.bridges is not None:
    for p in new_model.bridges.parameters():
        p.requires_grad = True

thermal_params = [p for p in new_model.parameters() if p.requires_grad]
print(f'Phase 2: {sum(p.numel() for p in thermal_params):,} trainable params')

optimizer_p2 = torch.optim.AdamW(thermal_params, lr=3e-5, weight_decay=0.01)

# Build difficulty-diverse training batches
difficulty_data = {
    'easy': [
        'Hello, how are you doing today?',
        'The sky is blue and the grass is',
        'One plus one equals',
        'Good morning! Today the weather is',
        'The cat is sleeping on the',
        'My favorite color is',
    ],
    'hard': [
        'Prove that there are infinitely many prime numbers using proof by contradiction.',
        'Implement a self-balancing AVL tree with insert, delete, and rebalance operations.',
        'Derive the Euler-Lagrange equation from the principle of stationary action.',
        'Design a distributed consensus algorithm that handles Byzantine faults with 3f+1 nodes.',
        'Prove the Cauchy-Schwarz inequality for an inner product space.',
        'Implement a concurrent garbage collector using a tri-color marking algorithm.',
    ],
}

# Tokenize difficulty data
diff_samples = []
for diff, texts in difficulty_data.items():
    for text in texts:
        inp, tgt = tokenize_text(text)
        if inp is not None:
            diff_samples.append((inp, tgt))
# Also include the domain data
for s in all_samples:
    diff_samples.append(s)

NUM_STEPS_P2 = 3000
pred_loss_ema = 0.0
losses_p2 = []

print(f'\nPhase 2: {NUM_STEPS_P2} steps (thermal + bridges, lambda_compute=0.05)')
for step in range(NUM_STEPS_P2):
    idx = rng.integers(0, len(diff_samples))
    inp, tgt = diff_samples[idx]

    losses = phase2_training_step(
        new_model, inp, tgt,
        lambda_compute=0.05,
        pred_loss_ema=pred_loss_ema,
    )

    (losses['total_loss'] / GRAD_ACCUM).backward()

    if (step + 1) % GRAD_ACCUM == 0:
        torch.nn.utils.clip_grad_norm_(thermal_params, 1.0)
        optimizer_p2.step()
        optimizer_p2.zero_grad(set_to_none=True)

    pred_val = losses['pred_loss'].item()
    pred_loss_ema = 0.99 * pred_loss_ema + 0.01 * pred_val if pred_loss_ema > 0 else pred_val

    losses_p2.append({
        'pred': pred_val,
        'tau': losses['tau'].item(),
        'tau_target': losses['tau_target'].item() if isinstance(losses['tau_target'], torch.Tensor) else losses['tau_target'],
        'strata': losses['num_strata_active'].item(),
    })

    if step % 500 == 0:
        recent = losses_p2[-500:] if len(losses_p2) >= 500 else losses_p2
        avg_pred = np.mean([r['pred'] for r in recent])
        avg_tau = np.mean([r['tau'] for r in recent])
        avg_strata = np.mean([r['strata'] for r in recent])
        print(f'  step {step:4d}/{NUM_STEPS_P2} | pred={avg_pred:.4f} \u03c4={avg_tau:.3f} strata={avg_strata:.1f}')

# Unfreeze all for future phases
for p in new_model.parameters():
    p.requires_grad = True

print(f'\nPhase 2 done. Final avg pred_loss: {np.mean([r["pred"] for r in losses_p2[-500:]]):.4f}')

---
## Phase 5 Retrain — Fix Rule Induction

Reinitialize HypothesisHead from scratch (with the `.detach()` fix in code). Train on ARC tasks.

In [ ]:
# Cell 12: Phase 5 Retrain — Fix Rule Induction
from flx.hypothesis import HypothesisHead
from flx.training.phase5_abstraction import phase5_training_step
from datasets import load_dataset

# Reinitialize hypothesis head from scratch (has .detach() fix)
fresh_hyp = HypothesisHead(d_model=d_model, hypothesis_dim=d_model).to(DEVICE)
new_model.attach_hypothesis_head(fresh_hyp)
new_model.train()
fresh_hyp.train()

# Load ARC training data
PAD_TOKEN = 0
SEP_TOKEN = 11

def flatten_grid_2d(grid):
    tokens = []
    for i, row in enumerate(grid):
        if i > 0:
            tokens.append(SEP_TOKEN)
        tokens.extend(v + 1 for v in row)
    return tokens

def arc_to_tensors(demos, test_pair, max_seq=SEQ_LEN):
    demo_inputs, demo_targets = [], []
    for pair in demos:
        inp_t = flatten_grid_2d(pair['input'])[:max_seq]
        out_t = flatten_grid_2d(pair['output'])[:max_seq]
        inp = F.pad(torch.tensor(inp_t, dtype=torch.long), (0, max_seq-len(inp_t)), value=PAD_TOKEN)
        out = F.pad(torch.tensor(out_t, dtype=torch.long), (0, max_seq-len(out_t)), value=-100)
        demo_inputs.append(inp)
        demo_targets.append(out)
    ti = flatten_grid_2d(test_pair['input'])[:max_seq]
    to_ = flatten_grid_2d(test_pair['output'])[:max_seq]
    ti = F.pad(torch.tensor(ti, dtype=torch.long), (0, max_seq-len(ti)), value=PAD_TOKEN)
    to_ = F.pad(torch.tensor(to_, dtype=torch.long), (0, max_seq-len(to_)), value=-100)
    return demo_inputs, demo_targets, ti, to_

# Load ARC train split
ds_train = load_dataset('lordspline/arc-agi', split='training')
arc_tasks = []
for row in ds_train:
    try:
        demos = row['train']
        tests = row['test']
        for tp in tests:
            di, dt, ti, to_ = arc_to_tensors(demos, tp)
            arc_tasks.append((di, dt, ti, to_))
    except (KeyError, TypeError, IndexError):
        continue
print(f'Loaded {len(arc_tasks)} ARC training tasks')

# Freeze trunk, router, merger
for p in new_model.shared_trunk.parameters():
    p.requires_grad = False
if new_model.thalamic_router is not None:
    for p in new_model.thalamic_router.parameters():
        p.requires_grad = False
for p in new_model.cortex_merger.parameters():
    p.requires_grad = False

# Optimizer: hypothesis head at full LR, rest at low LR
finetune_params = []
for cortex in new_model.cortices.values():
    finetune_params.extend([p for p in cortex.parameters() if p.requires_grad])
if new_model.bridges is not None:
    finetune_params.extend([p for p in new_model.bridges.parameters() if p.requires_grad])
if new_model.memory_controller is not None:
    finetune_params.extend([p for p in new_model.memory_controller.parameters() if p.requires_grad])
finetune_params.extend([p for p in new_model.decoder.parameters() if p.requires_grad])

optimizer_p5 = torch.optim.AdamW([
    {'params': fresh_hyp.parameters(), 'lr': 3e-4},
    {'params': finetune_params, 'lr': 1e-5},
])

NUM_STEPS_P5 = 2000
losses_p5 = []

print(f'\nPhase 5: {NUM_STEPS_P5} steps (hypothesis head + fine-tune)')
for step in range(NUM_STEPS_P5):
    idx = rng.integers(0, len(arc_tasks))
    di, dt, ti, to_ = arc_tasks[idx]
    di = [d.unsqueeze(0).to(DEVICE) for d in di]
    dt = [d.unsqueeze(0).to(DEVICE) for d in dt]
    ti = ti.unsqueeze(0).to(DEVICE)
    to_ = to_.unsqueeze(0).to(DEVICE)

    losses = phase5_training_step(
        new_model, fresh_hyp, di, dt, ti, to_,
        tau=0.8, max_loops=3, min_loops=1,
        lambda_cons=0.5, lambda_loop=0.01,
    )

    losses['total_loss'].backward()
    if (step + 1) % 2 == 0:
        torch.nn.utils.clip_grad_norm_(
            list(fresh_hyp.parameters()) + finetune_params, 1.0
        )
        optimizer_p5.step()
        optimizer_p5.zero_grad(set_to_none=True)

    losses_p5.append({
        'pred': losses['pred_loss'].item(),
        'cons': losses['consistency'].item(),
        'cons_target': losses['cons_target'].item(),
        'loops': losses['num_loops'].item(),
    })

    if step % 300 == 0:
        recent = losses_p5[-300:] if len(losses_p5) >= 300 else losses_p5
        avg_pred = np.mean([r['pred'] for r in recent])
        avg_cons = np.mean([r['cons'] for r in recent])
        print(f'  step {step:4d}/{NUM_STEPS_P5} | pred={avg_pred:.4f} cons={avg_cons:.3f}')

# Unfreeze all
for p in new_model.parameters():
    p.requires_grad = True

print(f'\nPhase 5 done. Final avg pred_loss: {np.mean([r["pred"] for r in losses_p5[-300:]]):.4f}')
print(f'Final avg consistency: {np.mean([r["cons"] for r in losses_p5[-300:]]):.3f}')

In [ ]:
# Cell 13: Save transplanted model
from pathlib import Path

SAVE_PATH = Path(f'{FLX_HUB}/nano_transplanted.flx')
print(f'Saving to {SAVE_PATH}...')

new_model.eval()
save_flx(new_model, str(SAVE_PATH))

# Verify
import yaml
with open(SAVE_PATH / 'manifest.yaml') as f:
    manifest = yaml.safe_load(f)
print(f'\nManifest:')
for k, v in manifest.items():
    print(f'  {k}: {v}')

print(f'\nDone! Model saved to {SAVE_PATH}')

---
## Full Validation Suite

Re-run all 8 experiments from the validation notebook on the transplanted model.

In [ ]:
# Cell 14: Full Validation Suite — re-run key experiments
new_model.eval()

def compute_loss(logits, targets):
    return F.cross_entropy(
        logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100
    ).item()

# === Experiment 2: Thermal Efficiency ===
print('=== Experiment 2: Thermal Efficiency ===')
difficulty_samples_val = {
    'easy': [
        'Hello, how are you doing today?',
        'The sky is blue and the grass is',
        'My name is Alice and I live in',
        'One plus one equals',
        'The color of the sun is',
        'Good morning! Today the weather is',
        'I like to eat pizza with',
        'The dog sat on the mat and',
        'print("hello world")',
        'She went to the store to buy',
    ],
    'hard': [
        'Prove that there are infinitely many prime numbers using proof by contradiction.',
        'Implement a self-balancing AVL tree with insert, delete, and rebalance operations.',
        'Derive the Euler-Lagrange equation from the principle of stationary action.',
        'Design a distributed consensus algorithm that handles Byzantine faults with 3f+1 nodes.',
        'Prove the Cauchy-Schwarz inequality for an inner product space.',
        'Analyze the computational complexity of the traveling salesman problem and explain why it is NP-hard.',
        'Derive the Navier-Stokes equations from conservation of momentum for a Newtonian fluid.',
        'Implement a concurrent garbage collector using a tri-color marking algorithm.',
        'Prove that the set of real numbers is uncountable using Cantor diagonal argument.',
        'Implement a type inference algorithm for a simply-typed lambda calculus.',
    ],
}

thermal_results = defaultdict(list)
with torch.no_grad():
    for difficulty, texts in difficulty_samples_val.items():
        for text in texts:
            inp, tgt = tokenize_text(text)
            if inp is None:
                continue
            trunk_out = new_model.shared_trunk(inp)
            tau = new_model.thermal_estimator(trunk_out).mean().item()
            active_strata = 0
            for cortex in new_model.cortices.values():
                for stratum in cortex.strata.values():
                    if tau >= stratum.tau_min:
                        active_strata += 1
            thermal_results[difficulty].append({'tau': tau, 'strata': active_strata})

easy_tau = np.mean([x['tau'] for x in thermal_results['easy']])
hard_tau = np.mean([x['tau'] for x in thermal_results['hard']])
easy_strata = np.mean([x['strata'] for x in thermal_results['easy']])
hard_strata = np.mean([x['strata'] for x in thermal_results['hard']])
print(f'  Easy: \u03c4={easy_tau:.3f}, strata={easy_strata:.1f}')
print(f'  Hard: \u03c4={hard_tau:.3f}, strata={hard_strata:.1f}')
print(f'  \u03c4 separation: easy={easy_tau:.3f} vs hard={hard_tau:.3f} (target: easy<0.45, hard>0.45)')
tau_pass = easy_tau < 0.45 and hard_tau > 0.45
print(f'  {"PASS" if tau_pass else "FAIL"}: Thermal Efficiency')

# === Experiment 0: Cortex Specialization ===
print(f'\n=== Experiment 0: Cortex Specialization ===')
routing_data = []
with torch.no_grad():
    for domain, texts in domain_texts.items():
        for text in texts:
            inp, _ = tokenize_text(text)
            if inp is None:
                continue
            trunk_out = new_model.shared_trunk(inp)
            scores = new_model.thalamic_router(trunk_out)
            score_dict = {k: v.mean().item() for k, v in scores.items()}
            routing_data.append((domain, score_dict))

cortex_names_list = list(new_model.cortex_names)
TOP_K = 50
spec_scores = {}
for cname in cortex_names_list:
    sorted_by_score = sorted(routing_data, key=lambda x: x[1].get(cname, 0), reverse=True)
    top_k = sorted_by_score[:TOP_K]
    domain_counts = defaultdict(int)
    for domain_label, _ in top_k:
        domain_counts[domain_label] += 1
    top_domain = max(domain_counts, key=domain_counts.get)
    spec = domain_counts[top_domain] / TOP_K
    spec_scores[cname] = spec
    print(f'  {cname:>12}: top_domain={top_domain:<10} spec={spec:.2f} counts={dict(domain_counts)}')

avg_spec = np.mean(list(spec_scores.values()))
print(f'  Avg spec: {avg_spec:.3f} (target: > 0.7)')
print(f'  {"PASS" if avg_spec > 0.7 else "MIXED" if avg_spec > 0.5 else "FAIL"}: Cortex Specialization')

# === Experiment 5: Abstract Rule Induction (quick check) ===
print(f'\n=== Experiment 5: Abstract Rule Induction (quick) ===')
if new_model.hypothesis_head is not None:
    ds_eval = load_dataset('lordspline/arc-agi', split='evaluation')
    eval_tasks = []
    for row in ds_eval:
        try:
            demos = row['train']
            tests = row['test']
            for tp in tests:
                di, dt, ti, to_ = arc_to_tensors(demos, tp)
                eval_tasks.append((di, dt, ti, to_))
        except (KeyError, TypeError, IndexError):
            continue

    N_EVAL = min(50, len(eval_tasks))
    results_eval = []
    with torch.no_grad():
        for i in range(N_EVAL):
            di, dt, ti, to_ = eval_tasks[i]
            di = [d.unsqueeze(0).to(DEVICE) for d in di]
            dt = [d.unsqueeze(0).to(DEVICE) for d in dt]
            ti = ti.unsqueeze(0).to(DEVICE)
            to_ = to_.unsqueeze(0).to(DEVICE)
            losses = phase5_training_step(
                new_model, new_model.hypothesis_head, di, dt, ti, to_,
                tau=0.8, max_loops=3, min_loops=1,
            )
            results_eval.append({
                'pred_loss': losses['pred_loss'].item(),
                'consistency': losses['consistency'].item(),
                'cons_target': losses['cons_target'].item(),
            })

    from scipy.stats import spearmanr
    cons_vals = [r['consistency'] for r in results_eval]
    neg_loss = [-r['pred_loss'] for r in results_eval]
    rho, pval = spearmanr(cons_vals, neg_loss)
    cons_target_diff = np.mean([abs(r['consistency'] - r['cons_target']) for r in results_eval])

    print(f'  Consistency-quality Spearman \u03c1: {rho:.3f} (p={pval:.4f}) (target: > 0.3)')
    print(f'  Cons tracks target: |cons-target| = {cons_target_diff:.3f} (target: < 0.15)')
    pass_p5 = rho > 0.3 and cons_target_diff < 0.15
    print(f'  {"PASS" if pass_p5 else "MIXED" if rho > 0.1 else "FAIL"}: Abstract Rule Induction')
else:
    print('  SKIP: No hypothesis head')

# === Summary ===
print(f'\n{"=" * 60}')
print('TRANSPLANT VALIDATION — SUMMARY')
print(f'{"=" * 60}')
print(f'  Thermal Efficiency: {"PASS" if tau_pass else "FAIL"}')
print(f'  Cortex Specialization: {"PASS" if avg_spec > 0.7 else "MIXED"}')
if new_model.hypothesis_head is not None:
    print(f'  Rule Induction: {"PASS" if pass_p5 else "MIXED" if rho > 0.1 else "FAIL"}')